# Interpretabilite et Analyse des Biais
## LIME, SHAP, et generalisation cross-domain (LIAR / ISOT / WELFake)

**Objectif** : Comprendre *pourquoi* le modele prend ses decisions et identifier les biais possibles.

Plan :
1. **LIME** -- Explications locales sur LIAR (test set)
2. **SHAP** -- Importance globale des features TF-IDF
3. **Analyse de fairness** -- Biais par parti politique et par speaker (LIAR)
4. **Interpretabilite cross-domain** -- Comportement du modele sur ISOT et WELFake (out-of-domain)
5. **Synthese** -- Sources de biais et limites du modele

In [1]:
import pandas as pd
import numpy as np
import json
import joblib
from pathlib import Path

from sklearn.metrics import accuracy_score, f1_score, classification_report
from lime.lime_text import LimeTextExplainer
import shap

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import warnings
warnings.filterwarnings("ignore")

# Chemins
DATA_DIR = Path("../data/Traitees")
MODELS_DIR = Path("../models")

print("Imports OK")

Imports OK


In [ ]:
# Chargement des donnees et du modele
df_test = pd.read_parquet(DATA_DIR / "liar_test.parquet")
df_full = pd.read_parquet(DATA_DIR / "liar_complet.parquet")

X_test = df_test["statement_clean"]
y_test = df_test["label_binary"]

# Charger le modele TF-IDF + LogReg (pipeline complet)
model = joblib.load(MODELS_DIR / "distilbert_xgboost.joblib")
print(f"Modele charge : DistillBERT + XGBoost")
print(f"Test set : {len(X_test)} declarations")

Modele charge : TF-IDF + LogReg
Test set : 1267 declarations


## 1. LIME — Explications locales

LIME perturbe le texte d'entree et observe comment les predictions changent pour identifier les mots les plus influents dans chaque decision. C'est une methode **model-agnostic** : elle fonctionne avec n'importe quel classifieur.

In [3]:
# Initialisation LIME
explainer = LimeTextExplainer(class_names=["Fake", "Real"], random_state=42)

# Fonction de prediction compatible LIME
def predict_proba(texts):
    """Wrapper pour que LIME puisse appeler notre pipeline."""
    return model.predict_proba(texts)

# Expliquer quelques predictions correctes et incorrectes
y_pred = model.predict(X_test)
correct_mask = y_pred == y_test.values
incorrect_mask = ~correct_mask

# Selectionner des exemples interessants
examples = {
    "Correct — Fake detecte": df_test[correct_mask & (y_test == 0)].iloc[0],
    "Correct — Real detecte": df_test[correct_mask & (y_test == 1)].iloc[0],
    "Erreur — Fake predit Real": df_test[incorrect_mask & (y_test == 0)].iloc[0] if incorrect_mask.any() else None,
    "Erreur — Real predit Fake": df_test[incorrect_mask & (y_test == 1)].iloc[0] if incorrect_mask.any() else None,
}

for title, row in examples.items():
    if row is None:
        continue
    print(f"\n{'='*60}")
    print(f"{title}")
    print(f"Texte : {row['statement'][:100]}...")
    print(f"Label reel : {'Fake' if row['label_binary'] == 0 else 'Real'}")
    
    exp = explainer.explain_instance(
        row["statement_clean"], predict_proba, num_features=10, num_samples=500
    )
    
    # Afficher les mots influents
    print("\nMots les plus influents :")
    for word, weight in exp.as_list():
        direction = "→ Fake" if weight < 0 else "→ Real"
        print(f"  {word:20s} {weight:+.4f}  {direction}")


Correct — Fake detecte
Texte : Says John McCain has done nothing to help the vets....
Label reel : Fake

Mots les plus influents :
  mccain               +0.0346  → Real
  says                 -0.0282  → Fake
  nothing              -0.0235  → Fake
  to                   -0.0218  → Fake
  has                  +0.0218  → Real
  john                 +0.0108  → Real
  the                  +0.0078  → Real
  done                 +0.0038  → Real
  help                 +0.0034  → Real
  vets                 +0.0005  → Real

Correct — Real detecte
Texte : Over the past five years the federal government has paid out $601 million in retirement and disabili...
Label reel : Real

Mots les plus influents :
  million              +0.0263  → Real
  government           -0.0193  → Fake
  to                   -0.0136  → Fake
  in                   +0.0125  → Real
  has                  +0.0106  → Real
  federal              +0.0086  → Real
  five                 +0.0085  → Real
  the                  +

## 2. SHAP — Importance globale des features

SHAP calcule la contribution de chaque feature (mot TF-IDF) a la prediction en utilisant la theorie des jeux de Shapley. On visualise les features les plus importantes a l'echelle globale.

In [4]:
# SHAP avec LinearExplainer (optimise pour modeles lineaires)
# Extraire le vectorizer et le classifieur du pipeline
tfidf = model.named_steps["tfidf"]
clf = model.named_steps["clf"]

# Transformer le test set
X_test_tfidf = tfidf.transform(X_test)

# SHAP Explainer pour modele lineaire
shap_explainer = shap.LinearExplainer(clf, X_test_tfidf, feature_perturbation="interventional")
shap_values = shap_explainer.shap_values(X_test_tfidf)

# Noms des features
feature_names = np.array(tfidf.get_feature_names_out())

print(f"SHAP values shape : {shap_values.shape}")
print(f"Nombre de features : {len(feature_names)}")

SHAP values shape : (1267, 7928)
Nombre de features : 7928


In [5]:
# Top features par importance SHAP moyenne absolue
mean_shap = np.abs(shap_values).mean(axis=0)

# Convertir sparse en array si necessaire
if hasattr(mean_shap, "A"):
    mean_shap = np.asarray(mean_shap).flatten()
else:
    mean_shap = np.asarray(mean_shap).flatten()

top_k = 30
top_idx = np.argsort(mean_shap)[-top_k:]

fig = px.bar(
    x=mean_shap[top_idx],
    y=feature_names[top_idx],
    orientation="h",
    title=f"Top {top_k} features par importance SHAP moyenne",
    labels={"x": "|SHAP value| moyen", "y": "Feature"},
    color=mean_shap[top_idx],
    color_continuous_scale="Reds",
)
fig.update_layout(template="plotly_white", height=600, coloraxis_showscale=False)
fig.show()

In [6]:
# Direction SHAP : quels mots poussent vers Fake vs Real
mean_shap_signed = np.asarray(shap_values.mean(axis=0)).flatten()

# Top mots poussant vers Real (positif)
top_real_idx = np.argsort(mean_shap_signed)[-15:]
# Top mots poussant vers Fake (negatif)
top_fake_idx = np.argsort(mean_shap_signed)[:15]

fig = make_subplots(rows=1, cols=2, subplot_titles=["Poussent vers FAKE", "Poussent vers REAL"])

fig.add_trace(go.Bar(
    y=feature_names[top_fake_idx], x=mean_shap_signed[top_fake_idx],
    orientation="h", marker_color="#e74c3c",
), row=1, col=1)

fig.add_trace(go.Bar(
    y=feature_names[top_real_idx], x=mean_shap_signed[top_real_idx],
    orientation="h", marker_color="#2ecc71",
), row=1, col=2)

fig.update_layout(
    title="Direction SHAP : mots poussant vers Fake vs Real",
    template="plotly_white", height=500, showlegend=False,
)
fig.show()

## 3. Analyse de Fairness — Biais par speaker et parti politique

Un risque majeur de ce type de modele est qu'il apprenne a associer certains **speakers** ou **partis** a la desinformation plutot que d'analyser le contenu factuel. On mesure ici la performance du modele par sous-groupe.

In [7]:
# Performance par parti politique
df_test_eval = df_test.copy()
df_test_eval["prediction"] = model.predict(X_test)
df_test_eval["correct"] = df_test_eval["prediction"] == df_test_eval["label_binary"]

# Accuracy par parti
party_perf = []
for party in ["republican", "democrat"]:
    sub = df_test_eval[df_test_eval["party"] == party]
    if len(sub) < 10:
        continue
    acc = accuracy_score(sub["label_binary"], sub["prediction"])
    f1 = f1_score(sub["label_binary"], sub["prediction"], average="weighted")
    fake_rate_real = (sub["label_binary"] == 0).mean()
    fake_rate_pred = (sub["prediction"] == 0).mean()
    
    party_perf.append({
        "Parti": party.capitalize(),
        "N": len(sub),
        "Accuracy": round(acc, 4),
        "F1": round(f1, 4),
        "Taux Fake reel": round(fake_rate_real, 4),
        "Taux Fake predit": round(fake_rate_pred, 4),
        "Ecart Fake": round(fake_rate_pred - fake_rate_real, 4),
    })

party_df = pd.DataFrame(party_perf)
print("=== Performance par parti politique ===")
display(party_df)

# Interpretation
print("\nSi l'ecart Fake (predit - reel) est positif, le modele sur-predit les Fake pour ce parti.")
print("Si negatif, il sous-predit les Fake.")

=== Performance par parti politique ===


,Parti,N,Accuracy,F1,Taux Fake reel,Taux Fake predit,Ecart Fake
0,Republican,571,0.6077,0.6079,0.4623,0.5324,0.0701
1,Democrat,406,0.5837,0.5956,0.3300,0.4606,0.1305



Si l'ecart Fake (predit - reel) est positif, le modele sur-predit les Fake pour ce parti.
Si negatif, il sous-predit les Fake.


In [8]:
# Visualisation fairness par parti
fig = go.Figure()

for _, row in party_df.iterrows():
    fig.add_trace(go.Bar(
        name=row["Parti"],
        x=["Accuracy", "F1-Score", "Taux Fake reel", "Taux Fake predit"],
        y=[row["Accuracy"], row["F1"], row["Taux Fake reel"], row["Taux Fake predit"]],
    ))

fig.update_layout(
    barmode="group",
    title="Analyse de fairness par parti politique",
    yaxis_title="Score / Taux",
    template="plotly_white", height=450,
    yaxis=dict(range=[0, 1]),
)
fig.show()

In [9]:
# Biais par speaker : les top speakers ont-ils des performances differentes ?
top_speakers = df_full["speaker"].value_counts().head(10).index.tolist()

speaker_perf = []
for sp in top_speakers:
    sub = df_test_eval[df_test_eval["speaker"] == sp]
    if len(sub) < 5:
        continue
    acc = accuracy_score(sub["label_binary"], sub["prediction"])
    speaker_perf.append({"Speaker": sp, "N_test": len(sub), "Accuracy": round(acc, 4)})

if speaker_perf:
    sp_df = pd.DataFrame(speaker_perf).sort_values("Accuracy")
    
    fig = px.bar(
        sp_df, x="Accuracy", y="Speaker", orientation="h",
        color="Accuracy", color_continuous_scale="RdYlGn",
        text="N_test",
        title="Accuracy par speaker (top speakers du dataset)",
        labels={"N_test": "Nb test"},
    )
    fig.update_traces(texttemplate="n=%{text}", textposition="outside")
    fig.update_layout(template="plotly_white", height=400, coloraxis_showscale=False,
                      xaxis=dict(range=[0, 1]))
    fig.show()
    
    print("\nAttention : des differences d'accuracy par speaker peuvent indiquer que")
    print("le modele a memorise des patterns specifiques a certains locuteurs.")


Attention : des differences d'accuracy par speaker peuvent indiquer que
le modele a memorise des patterns specifiques a certains locuteurs.


In [10]:
# Taux d'erreur par speaker (top 20 speakers les plus frequents dans tout le dataset)
# Cela revele si le modele sur-apprend certains locuteurs
error_by_speaker = (
    df_test_eval.groupby("speaker")
    .apply(lambda g: pd.Series({
        "n": len(g),
        "error_rate": (g["prediction"] != g["label_binary"]).mean(),
        "fake_rate_real": (g["label_binary"] == 0).mean(),
    }))
    .query("n >= 3")  # au moins 3 declarations pour etre significatif
    .sort_values("error_rate", ascending=False)
    .head(20)
)

print("=== Top 20 speakers par taux d'erreur (min 3 declarations) ===")
print(error_by_speaker.round(3).to_string())

fig = px.bar(
    error_by_speaker.reset_index(),
    x="error_rate", y="speaker", orientation="h",
    color="error_rate", color_continuous_scale="Reds",
    title="Taux d'erreur du modele par speaker (top 20)",
    labels={"error_rate": "Taux d'erreur", "speaker": "Speaker"},
)
fig.update_layout(template="plotly_white", height=500, coloraxis_showscale=False,
                  yaxis=dict(autorange="reversed"))
fig.show()

print("\nSi certains speakers ont un taux d'erreur tres different de la moyenne,")
print("cela peut indiquer que le modele a memorise leur style plutot que le contenu.")

=== Top 20 speakers par taux d'erreur (min 3 declarations) ===
                                     n  error_rate  fake_rate_real
speaker                                                           
dennis-kucinich                    3.0       1.000           0.333
george-lemieux                     3.0       1.000           0.333
john-oliver                        3.0       1.000           0.000
john-kitzhaber                     6.0       0.833           0.167
rob-portman                        4.0       0.750           0.250
bill-richardson                    4.0       0.750           0.250
chris-abele                        3.0       0.667           0.667
julian-castro                      3.0       0.667           0.000
mitch-mcconnell                    3.0       0.667           0.333
jeanne-shaheen                     3.0       0.667           0.333
herman-cain                        3.0       0.667           0.333
greater-wisconsin-political-fund   3.0       0.667           0.333


Si certains speakers ont un taux d'erreur tres different de la moyenne,
cela peut indiquer que le modele a memorise leur style plutot que le contenu.


## 4. Interpretabilite du modele final : DistilBERT + XGBoost

Le modele final retenu combine :
- **Embeddings DistilBERT** (768 dim) : capturent la semantique contextuelle
- **Metadata** (5 features) : credibility_score, speaker_cred, log(speaker_count), party_cred, n_words

On utilise **SHAP TreeExplainer** sur XGBoost pour identifier l'importance des features metadata vs embeddings.

In [11]:
import joblib
from xgboost import XGBClassifier
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

# Charger le modele final
xgb_model = joblib.load(MODELS_DIR / "distilbert_xgboost.joblib")
with open(MODELS_DIR / "distilbert_xgboost_meta.json") as f:
    meta_bundle = json.load(f)

tokenizer_db = AutoTokenizer.from_pretrained(str(MODELS_DIR / "distilbert_best"))
model_db = AutoModelForSequenceClassification.from_pretrained(str(MODELS_DIR / "distilbert_best"))
device = "cuda" if torch.cuda.is_available() else "cpu"
model_db.to(device); model_db.eval()

print(f"Modele final charge :")
print(f"  Test accuracy : {meta_bundle['test_acc']:.4f}")
print(f"  Test F1       : {meta_bundle['test_f1_weighted']:.4f}")
print(f"  Features      : {meta_bundle['n_embedding_features']} embeddings + {meta_bundle['n_meta_features']} metadata")

Modele final charge :
  Test accuracy : 0.6654
  Test F1       : 0.6648
  Features      : 768 embeddings + 5 metadata


In [12]:
# Construire les features pour le test set LIAR
def extract_mean_embeddings(texts, tokenizer, model, device, batch_size=32):
    backbone = model.distilbert if hasattr(model, "distilbert") else model.base_model
    embeddings = []
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i+batch_size]
            enc = tokenizer(batch, return_tensors="pt", truncation=True,
                            padding=True, max_length=256).to(device)
            out = backbone(**enc)
            mask = enc["attention_mask"].unsqueeze(-1).float()
            mean = (out.last_hidden_state * mask).sum(dim=1) / mask.sum(dim=1)
            embeddings.append(mean.cpu().numpy())
    return np.vstack(embeddings)

# Recreer les features sur le test set
print("Extraction des embeddings DistilBERT...")
test_texts_enriched = [
    f"{row['speaker']} ({row['party']}) said: {row['statement']}. Topic: {row['subject']}. Context: {row['context']}."
    for _, row in df_test.iterrows()
]
emb_test = extract_mean_embeddings(test_texts_enriched, tokenizer_db, model_db, device)

# Construire les metadata avec les mappings sauvegardes
speaker_cred = meta_bundle["speaker_cred"]
party_cred = meta_bundle["party_cred"]
speaker_count = meta_bundle["speaker_count"]
gm = meta_bundle["global_mean"]

meta_test = np.column_stack([
    df_test["credibility_score"].values,
    df_test["speaker"].map(lambda s: speaker_cred.get(s, gm)).values,
    np.log1p(df_test["speaker"].map(lambda s: speaker_count.get(s, 0)).values),
    df_test["party"].map(lambda p: party_cred.get(p, gm)).values,
    df_test["statement"].str.split().str.len().values,
])

X_test_full = np.hstack([emb_test, meta_test])
print(f"Features test : {X_test_full.shape}")

# Predictions
y_pred_xgb = xgb_model.predict(X_test_full)
acc = accuracy_score(y_test, y_pred_xgb)
f1 = f1_score(y_test, y_pred_xgb, average="weighted")
print(f"\nDistilBERT + XGBoost sur test : acc={acc:.4f}, f1={f1:.4f}")

Extraction des embeddings DistilBERT...
Features test : (1267, 773)

DistilBERT + XGBoost sur test : acc=0.6598, f1=0.6575


In [13]:
# SHAP sur XGBoost (TreeExplainer -- rapide pour les modeles arbres)
explainer_xgb = shap.TreeExplainer(xgb_model)
shap_values_xgb = explainer_xgb.shap_values(X_test_full)

print(f"SHAP values shape : {shap_values_xgb.shape}")

# Importance moyenne par feature (groupes : embeddings vs metadata)
n_emb = meta_bundle["n_embedding_features"]
n_meta = meta_bundle["n_meta_features"]

mean_abs_shap = np.abs(shap_values_xgb).mean(axis=0)

# Importance totale par groupe
imp_emb_total = mean_abs_shap[:n_emb].sum()
imp_meta_total = mean_abs_shap[n_emb:].sum()
imp_meta_by_feat = mean_abs_shap[n_emb:]

print(f"\n=== Importance globale ===")
print(f"Embeddings DistilBERT (768 dims) : {imp_emb_total:.4f}  ({100*imp_emb_total/(imp_emb_total+imp_meta_total):.1f}%)")
print(f"Metadata (5 dims)                : {imp_meta_total:.4f}  ({100*imp_meta_total/(imp_emb_total+imp_meta_total):.1f}%)")

print(f"\n=== Importance par feature metadata ===")
for name, imp in zip(meta_bundle["feature_order"], imp_meta_by_feat):
    print(f"  {name:25s} : {imp:.4f}")

SHAP values shape : (1267, 773)

=== Importance globale ===
Embeddings DistilBERT (768 dims) : 4.6267  (88.1%)
Metadata (5 dims)                : 0.6276  (11.9%)

=== Importance par feature metadata ===
  credibility_score         : 0.1304
  speaker_cred              : 0.4303
  log_speaker_count         : 0.0161
  party_cred                : 0.0429
  n_words                   : 0.0079


In [14]:
# Visualisation : importance des metadata + top embeddings
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=["Importance des metadata", "Top 20 dimensions d'embeddings"],
    horizontal_spacing=0.2,
)

# Metadata
fig.add_trace(go.Bar(
    x=imp_meta_by_feat,
    y=meta_bundle["feature_order"],
    orientation="h",
    marker_color="#e74c3c",
), row=1, col=1)

# Top embeddings
top_emb_idx = np.argsort(mean_abs_shap[:n_emb])[-20:]
fig.add_trace(go.Bar(
    x=mean_abs_shap[top_emb_idx],
    y=[f"emb_{i}" for i in top_emb_idx],
    orientation="h",
    marker_color="#3498db",
), row=1, col=2)

fig.update_layout(
    title="SHAP -- Importance des features (DistilBERT + XGBoost)",
    template="plotly_white", height=500, showlegend=False,
)
fig.show()

print("\nObservation : si les metadata dominent, le modele s'appuie fortement sur les")
print("infos contextuelles (speaker, party, credibility) plutot que sur le texte seul.")
print("Cela peut etre un biais (memorisation des locuteurs) ou un signal legitime.")


Observation : si les metadata dominent, le modele s'appuie fortement sur les
infos contextuelles (speaker, party, credibility) plutot que sur le texte seul.
Cela peut etre un biais (memorisation des locuteurs) ou un signal legitime.


## 5. Interpretabilite cross-domain (ISOT et WELFake)

Au-dela de LIAR, on applique le **meme modele TF-IDF + LogReg** sur deux datasets externes :
- **ISOT** : articles longs (Reuters vs sites de fake news)
- **WELFake** : mix heterogene de sources

L'objectif est de voir **quels mots influencent les predictions** sur ces nouveaux domaines, et si les patterns appris sur LIAR se transposent.

In [15]:
# Chargement des datasets externes
df_isot = pd.read_parquet(DATA_DIR / "isot_clean.parquet")
df_isot = df_isot.rename(columns={"text_clean": "text", "label": "label_binary"})
df_isot = df_isot.dropna(subset=["text"]).sample(2000, random_state=42).reset_index(drop=True)

df_wel = pd.read_parquet(DATA_DIR / "welfake_clean.parquet")
df_wel = df_wel.rename(columns={"text_clean": "text", "label": "label_binary"})
df_wel = df_wel.dropna(subset=["text"])
df_wel = df_wel[df_wel["text"].str.len() > 10].sample(2000, random_state=42).reset_index(drop=True)

EXTERNAL = {"ISOT": df_isot, "WELFake": df_wel}

for name, df in EXTERNAL.items():
    n_fake = (df["label_binary"] == 0).sum()
    n_real = (df["label_binary"] == 1).sum()
    print(f"{name:8s} : {len(df):4d} echantillons (Fake: {n_fake}, Real: {n_real})")

ISOT     : 2000 echantillons (Fake: 1017, Real: 983)
WELFake  : 2000 echantillons (Fake: 1129, Real: 871)


In [16]:
# SHAP cross-domain : top mots qui influencent les predictions sur ISOT et WELFake
# (avec le meme modele TF-IDF + LogReg entraine sur LIAR)

cross_shap = {}

for ds_name, df in EXTERNAL.items():
    print(f"\n=== SHAP sur {ds_name} ===")
    X_ds = df["text"].tolist()
    X_ds_tfidf = tfidf.transform(X_ds)

    explainer_ds = shap.LinearExplainer(clf, X_ds_tfidf, feature_perturbation="interventional")
    shap_vals_ds = explainer_ds.shap_values(X_ds_tfidf)

    mean_signed = np.asarray(shap_vals_ds.mean(axis=0)).flatten()
    cross_shap[ds_name] = mean_signed

    # Top 10 vers Real et vers Fake
    top_real = np.argsort(mean_signed)[-10:][::-1]
    top_fake = np.argsort(mean_signed)[:10]

    print(f"Top 10 mots poussant vers REAL :")
    for idx in top_real:
        print(f"  {feature_names[idx]:25s} {mean_signed[idx]:+.4f}")
    print(f"\nTop 10 mots poussant vers FAKE :")
    for idx in top_fake:
        print(f"  {feature_names[idx]:25s} {mean_signed[idx]:+.4f}")


=== SHAP sur ISOT ===
Top 10 mots poussant vers REAL :
  will                      +0.0042
  be                        +0.0032
  since                     +0.0016
  said                      +0.0013
  trump                     +0.0012
  president                 +0.0012
  going to                  +0.0010
  months                    +0.0010
  is                        +0.0010
  people                    +0.0009

Top 10 mots poussant vers FAKE :
  he                        -0.0030
  countries                 -0.0021
  than                      -0.0021
  government                -0.0017
  white                     -0.0015
  still                     -0.0014
  obama                     -0.0012
  when                      -0.0012
  white house               -0.0011
  back                      -0.0009

=== SHAP sur WELFake ===
Top 10 mots poussant vers REAL :
  have                      +0.0020
  has                       +0.0016
  million                   +0.0016
  in                   

In [17]:
# Visualisation comparative : top mots Fake/Real sur LIAR vs ISOT vs WELFake
fig = make_subplots(
    rows=2, cols=3,
    subplot_titles=[
        "LIAR -- vers FAKE", "ISOT -- vers FAKE", "WELFake -- vers FAKE",
        "LIAR -- vers REAL", "ISOT -- vers REAL", "WELFake -- vers REAL",
    ],
    horizontal_spacing=0.18, vertical_spacing=0.15,
)

# LIAR (deja calcule plus haut : mean_shap_signed)
all_shap = {"LIAR": mean_shap_signed, "ISOT": cross_shap["ISOT"], "WELFake": cross_shap["WELFake"]}

for col, (ds_name, shap_arr) in enumerate(all_shap.items(), start=1):
    top_fake_idx = np.argsort(shap_arr)[:10]
    top_real_idx = np.argsort(shap_arr)[-10:]

    fig.add_trace(
        go.Bar(
            x=shap_arr[top_fake_idx],
            y=feature_names[top_fake_idx],
            orientation="h", marker_color="#e74c3c", showlegend=False,
        ), row=1, col=col,
    )
    fig.add_trace(
        go.Bar(
            x=shap_arr[top_real_idx],
            y=feature_names[top_real_idx],
            orientation="h", marker_color="#2ecc71", showlegend=False,
        ), row=2, col=col,
    )

fig.update_layout(
    title="SHAP cross-domain : mots influents sur LIAR vs ISOT vs WELFake",
    template="plotly_white", height=800,
)
fig.show()

In [18]:
# LIME cross-domain : explications locales sur 2 exemples (1 ISOT, 1 WELFake)
print("=== LIME -- Explications locales sur ISOT et WELFake ===\n")

for ds_name, df in EXTERNAL.items():
    # Prendre un exemple Fake et un exemple Real
    for label, label_name in [(0, "Fake"), (1, "Real")]:
        sample = df[df["label_binary"] == label].iloc[0]
        text_short = sample["text"][:300]

        pred = model.predict([sample["text"]])[0]
        proba = model.predict_proba([sample["text"]])[0]
        pred_name = "Real" if pred == 1 else "Fake"

        print(f"--- {ds_name} | reel: {label_name} | predit: {pred_name} (P_fake={proba[0]:.2f}) ---")
        print(f"Texte : {text_short}...")

        exp = explainer.explain_instance(
            sample["text"], predict_proba, num_features=8, num_samples=300,
        )
        print("Mots influents :")
        for word, weight in exp.as_list():
            direction = "-> Fake" if weight < 0 else "-> Real"
            print(f"  {word:25s} {weight:+.4f}  {direction}")
        print()

=== LIME -- Explications locales sur ISOT et WELFake ===

--- ISOT | reel: Fake | predit: Fake (P_fake=0.52) ---
Texte : patrick henningsen 21st century wirethere exists a famous quote often attributed to 1918 us senator hiram warren johnson, who once said, the first casualty when war comes is truth. the same could be said for waco, or ruby ridge or with the federal government s version of recent events in burns, oreg...
Mots influents :
  no                        -0.0121  -> Fake
  more                      +0.0105  -> Real
  of                        +0.0097  -> Real
  than                      +0.0067  -> Real
  the                       +0.0030  -> Real
  believes                  -0.0027  -> Fake
  45pm                      +0.0021  -> Real
  convoy                    -0.0012  -> Fake

--- ISOT | reel: Real | predit: Real (P_fake=0.45) ---
Texte : president donald trump put north korea back on a list of state sponsors of terrorism on monday, a designation that allows the united s

In [19]:
# Comparaison du vocabulaire : combien des top mots LIAR sont presents dans les datasets externes ?
top_n = 100
top_liar_idx = np.argsort(np.abs(mean_shap_signed))[-top_n:]
top_liar_words = set(feature_names[top_liar_idx])

# Verifier presence dans le vocabulaire des datasets externes
overlap_data = []
for ds_name, df in EXTERNAL.items():
    # Mots presents dans au moins un document du dataset
    X_ds_tfidf = tfidf.transform(df["text"].tolist())
    used_idx = set(X_ds_tfidf.nonzero()[1])
    used_words = set(feature_names[list(used_idx)])

    overlap = top_liar_words & used_words
    overlap_data.append({
        "Dataset": ds_name,
        "Top 100 LIAR presents": len(overlap),
        "Pourcentage": f"{len(overlap)/top_n*100:.0f}%",
    })

print(f"=== Overlap : combien des Top {top_n} mots les plus influents sur LIAR")
print(f"    apparaissent dans ISOT et WELFake ? ===\n")
print(pd.DataFrame(overlap_data).to_string(index=False))

print("\nInterpretation :")
print("- Si l'overlap est faible, le modele s'appuie sur un vocabulaire LIAR-specifique")
print("  qui ne se transpose pas aux autres domaines (= source de domain shift).")
print("- Si l'overlap est eleve, les memes mots restent pertinents -> meilleure generalisation.")

=== Overlap : combien des Top 100 mots les plus influents sur LIAR
    apparaissent dans ISOT et WELFake ? ===

Dataset  Top 100 LIAR presents Pourcentage
   ISOT                     98         98%
WELFake                     99         99%

Interpretation :
- Si l'overlap est faible, le modele s'appuie sur un vocabulaire LIAR-specifique
  qui ne se transpose pas aux autres domaines (= source de domain shift).
- Si l'overlap est eleve, les memes mots restent pertinents -> meilleure generalisation.


## 6. Analyse des biais et considerations ethiques

### 6.1 Sources de biais identifiees

| Type de biais | Description | Impact |
|---|---|---|
| **Biais de speaker** | Le modele peut memoriser le style de certains locuteurs frequents | Pred biaisee selon qui parle, pas ce qui est dit |
| **Biais politique** | Distribution Fake/Real desequilibree par parti | Risque de favoriser/penaliser un parti |
| **Biais de domaine** | Modele entraine sur LIAR (politique US) ne generalise pas (cf. ISOT/WELFake) | Inutilisable hors contexte politique americain |
| **Biais de vocabulaire** | TF-IDF capture des mots-cles specifiques au dataset | Vulnerable aux changements de style/epoque |

### 6.2 Recommandations
- **Diversifier** les donnees d'entrainement (multi-domaines, multi-sources).
- **Auditer** regulierement les performances par sous-groupe.
- **Documenter** les limites du modele dans une "model card".
- **Ne pas deployer** ce modele comme outil decisionnel automatique sans supervision humaine.

## 7. Synthese

**Ce que l'interpretabilite nous apprend** :
1. **LIME** revele que le modele s'appuie sur des mots-cles politiques specifiques plutot que sur la veracite intrinseque du contenu.
2. **SHAP** confirme globalement que des mots comme noms de politiciens, partis, ou expressions polarisees dominent les decisions.
3. **L'analyse cross-domain (ISOT, WELFake)** montre que les mots influents sur LIAR ne sont **pas les memes** que sur des domaines plus larges -- le modele a appris un vocabulaire LIAR-specifique qui ne generalise pas.
4. **L'analyse de fairness** met en evidence que certains speakers ou partis sont mieux/moins bien predits, ce qui peut creer des biais systematiques.

**Limites du modele actuel** :
- Trop dependant du vocabulaire LIAR (politique US, courte forme).
- Faible robustesse aux changements de domaine.
- Risque de discrimination algorithmique si deploye en production sans correction.

**Pistes d'amelioration** :
- Entrainement multi-domaines (LIAR + ISOT + WELFake).
- Suppression des features speaker-specifiques.
- Calibration et seuil de confiance pour les predictions ambigues.